In [0]:

import pandas as pd
import os

# Mostrar la carpeta desde donde estamos trabajando.
print("Carpeta actual:", os.getcwd())

# Leer la base original de ROGA.
df = pd.read_csv("datathon_dataset_entrega.csv")

# Mostrar las dimensiones del archivo.
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

# Mostrar los primeros 5 registros.
display(df.head())


In [0]:

# Revisar los nombres de todas las columnas
print("COLUMNAS DE LA BASE DE DATOS")
print("-" * 40)

for columna in df.columns:
    print(columna)

# Revisar qué tipo de información tiene cada columna
print("\nTIPOS DE DATOS")
print("-" * 40)

df.info()

# Revisar cuántos valores faltan por columna
print("\nVALORES FALTANTES")
print("-" * 40)

display(df.isnull().sum().to_frame("Valores faltantes"))


In [0]:

# Identificar columnas que contienen valores faltantes
faltantes = df.isnull().sum()

# Mostrar solamente las columnas con valores faltantes
faltantes = faltantes[faltantes > 0]

print("COLUMNAS CON DATOS FALTANTES")
print("-" * 40)

for columna, cantidad in faltantes.items():
    porcentaje = (cantidad / len(df)) * 100

    print(f"Columna: {columna}")
    print(f"Valores faltantes: {cantidad}")
    print(f"Porcentaje: {porcentaje:.2f}%")
    print("-" * 40)


In [0]:

# Revisar los tipos de cliente existentes
print("TIPOS DE CLIENTE")
print("-" * 40)

print(df["Tipo_Cliente"].value_counts(dropna=False))

# Revisar cómo se relacionan con el cumplimiento
print("\nCUMPLIMIENTO POR TIPO DE CLIENTE")
print("-" * 40)

tabla = pd.crosstab(
    df["Tipo_Cliente"],
    df["cumple_plazo"],
    margins=True
)

display(tabla)

# Mostrar el porcentaje de cumplimiento
porcentajes = pd.crosstab(
    df["Tipo_Cliente"],
    df["cumple_plazo"],
    normalize="index"
) * 100

display(porcentajes.round(2))


In [0]:

# Crear una copia de los datos originales
df_limpio = df.copy()

# Eliminar espacios al principio y al final
df_limpio["Tipo_Cliente"] = (
    df_limpio["Tipo_Cliente"]
    .astype("string")
    .str.strip()
)

# Seleccionar únicamente los clientes nuevos
df_nuevos = df_limpio[
    df_limpio["Tipo_Cliente"] == "Cliente Nuevo"
].copy()

print("CLIENTES NUEVOS")
print("-" * 40)

print("Total de registros:", len(df_nuevos))

print("\nResultados de cumplimiento:")
print(df_nuevos["cumple_plazo"].value_counts())

# Mostrar datos solamente si encontramos registros
if len(df_nuevos) > 0:
    display(df_nuevos.head())
else:
    print("No se encontraron clientes nuevos.")


In [0]:

# Revisar los tipos de datos de clientes nuevos
print("TIPOS DE VARIABLES")
print("-" * 40)

print(df_nuevos.dtypes)

# Resumen estadístico de variables numéricas
print("\nRESUMEN ESTADÍSTICO")
print("-" * 40)

display(df_nuevos.describe().T)

# Identificar variables categóricas
categoricas = df_nuevos.select_dtypes(
    include=["object", "string", "category"]
)

print("\nVARIABLES CATEGÓRICAS")
print("-" * 40)

for columna in categoricas.columns:
    print(f"\n{columna}")
    print(df_nuevos[columna].value_counts(dropna=False).head(10))


In [0]:

# Mostrar las categorías de cada variable de texto
columnas_categoricas = df_nuevos.select_dtypes(
    include=["object", "string", "category"]
).columns

for columna in columnas_categoricas:
    print("\n" + "=" * 45)
    print("VARIABLE:", columna)
    print("=" * 45)

    resumen = df_nuevos[columna].value_counts(
        dropna=False
    ).reset_index()

    resumen.columns = ["Categoria", "Cantidad"]

    resumen["Porcentaje"] = (
        resumen["Cantidad"] / len(df_nuevos) * 100
    ).round(2)

    display(resumen)


In [0]:

import matplotlib.pyplot as plt

# Contar los resultados de los clientes nuevos
conteo = df_nuevos["cumple_plazo"].value_counts()

# Crear la gráfica
plt.figure(figsize=(8, 5))

plt.bar(
    ["Pagaron a tiempo", "Fuera de plazo"],
    [conteo[1], conteo[0]]
)

# Agregar valores encima de cada barra
for i, valor in enumerate([conteo[1], conteo[0]]):
    plt.text(i, valor + 15, str(valor), ha="center")

plt.title("Cumplimiento de crédito - Clientes nuevos")
plt.ylabel("Número de créditos")
plt.ylim(0, max(conteo) * 1.15)

plt.show()


In [0]:

import matplotlib.pyplot as plt

# Crear grupos según el cumplimiento
pagan = df_nuevos[
    df_nuevos["cumple_plazo"] == 1
]["Capacidad_Quincenal"].dropna()

no_pagan = df_nuevos[
    df_nuevos["cumple_plazo"] == 0
]["Capacidad_Quincenal"].dropna()

# Revisar estadísticas antes de graficar
print("CAPACIDAD QUINCENAL POR GRUPO")
print("-" * 40)

print("Mediana de quienes cumplen:", pagan.median())
print("Mediana de quienes no cumplen:", no_pagan.median())

# Crear gráfica comparativa
plt.figure(figsize=(9, 5))

plt.boxplot(
    [pagan, no_pagan],
    tick_labels=["A tiempo", "Fuera de plazo"],
    showfliers=False
)

plt.title("Capacidad quincenal según cumplimiento")
plt.ylabel("Capacidad quincenal")
plt.grid(axis="y", alpha=0.3)

plt.show()


In [0]:

import pandas as pd
import matplotlib.pyplot as plt

# Crear una copia para el análisis
analisis = df_nuevos[
    ["Capacidad_Quincenal", "cumple_plazo"]
].dropna().copy()

# Dividir los clientes en 5 grupos de tamaño similar
analisis["Grupo"] = pd.qcut(
    analisis["Capacidad_Quincenal"],
    q=5,
    duplicates="drop"
)

# Calcular porcentaje de incumplimiento por grupo
resumen = analisis.groupby(
    "Grupo", observed=True
).agg(
    Clientes=("cumple_plazo", "size"),
    Cumplimiento=("cumple_plazo", "mean"),
    Capacidad_Mediana=("Capacidad_Quincenal", "median")
).reset_index()

resumen["Incumplimiento"] = (
    1 - resumen["Cumplimiento"]
) * 100

# Crear etiquetas sencillas
resumen["Nivel"] = [
    f"Grupo {i+1}" for i in range(len(resumen))
]

display(resumen)

# Graficar
plt.figure(figsize=(9, 5))

barras = plt.bar(
    resumen["Nivel"],
    resumen["Incumplimiento"]
)

for barra in barras:
    valor = barra.get_height()
    plt.text(
        barra.get_x() + barra.get_width()/2,
        valor + 0.5,
        f"{valor:.1f}%",
        ha="center"
    )

plt.title("Incumplimiento según capacidad quincenal")
plt.xlabel("Capacidad quincenal: menor a mayor")
plt.ylabel("Créditos fuera de plazo (%)")
plt.ylim(0, 100)
plt.grid(axis="y", alpha=0.2)

plt.show()


In [0]:

# Investigar cómo se relaciona la capacidad
# quincenal con otras variables económicas.

columnas = [
    "Capacidad_Quincenal",
    "pago_quincenal",
    "Ingreso_Cliente",
    "monto_financiado"
]

# Resumen estadístico
display(df_nuevos[columnas].describe().T)

# Correlaciones entre las variables
print("CORRELACIONES")
display(
    df_nuevos[columnas]
    .corr(method="spearman")
    .round(3)
)

# Revisar capacidades negativas, cero y positivas
capacidad = df_nuevos["Capacidad_Quincenal"]

print("\nDISTRIBUCIÓN DE CAPACIDAD")
print("Negativa:", (capacidad < 0).sum())
print("Cero:", (capacidad == 0).sum())
print("Positiva:", (capacidad > 0).sum())


In [0]:

# Revisar todas las columnas disponibles
print("COLUMNAS DISPONIBLES")
print("-" * 45)

for columna in df_nuevos.columns:
    print(columna)

# Buscar columnas relacionadas con fechas
columnas_fecha = [
    col for col in df_nuevos.columns
    if "fecha" in col.lower()
]

print("\nCOLUMNAS DE FECHA")
print(columnas_fecha)

# Revisar sus valores originales
for columna in columnas_fecha:
    print(f"\nVariable: {columna}")
    print("Tipo:", df_nuevos[columna].dtype)
    print("Primeros valores:")
    print(df_nuevos[columna].head(10).to_string(index=False))

# Verificar etiquetas del modelo principal
print("\nETIQUETAS DE CUMPLIMIENTO")
print(df_nuevos["cumple_plazo"].value_counts(dropna=False))


In [0]:

import pandas as pd

# Crear copia para no modificar los datos originales
df_modelo = df_nuevos.copy()

# Convertir fechas de texto a formato datetime
df_modelo["fecha_entrega"] = pd.to_datetime(
    df_modelo["fecha_entrega"],
    errors="coerce"
)

# Comprobar fechas
print("ANÁLISIS TEMPORAL")
print("-" * 40)

print("Fecha más antigua:",
      df_modelo["fecha_entrega"].min())

print("Fecha más reciente:",
      df_modelo["fecha_entrega"].max())

print("Fechas faltantes:",
      df_modelo["fecha_entrega"].isna().sum())

# Mostrar cantidad de créditos por año
df_modelo["anio"] = df_modelo["fecha_entrega"].dt.year

print("\nCRÉDITOS POR AÑO")
display(
    df_modelo["anio"]
    .value_counts()
    .sort_index()
    .rename_axis("Año")
    .reset_index(name="Créditos")
)

# Ordenar cronológicamente
df_modelo = df_modelo.sort_values(
    "fecha_entrega"
).reset_index(drop=True)

print("\nPrimer crédito:", df_modelo["fecha_entrega"].iloc[0])
print("Último crédito:", df_modelo["fecha_entrega"].iloc[-1])


In [0]:

import pandas as pd

# Usar los clientes nuevos ya ordenados por fecha
datos = df_modelo.copy()

# Variables disponibles al solicitar el crédito
numericas = [
    "Importe_Pedido",
    "pago_quincenal",
    "monto_financiado",
    "Anticipo_Condicionado",
    "ratio_anticipo",
    "plazo_meses",
    "n_productos",
    "Ingreso_Cliente",
    "Meses_Empleo",
    "Capacidad_Quincenal",
    "personas_en_Domicilio",
    "sueldo_conyuge",
    "meses_antiguedad_conyuge",
    "Tiene_Aval"
]

categoricas = [
    "Tipo_Domicilio",
    "Tipo_Empleo",
    "Tipo_Empleo_Conyuge"
]

variables = numericas + categoricas

# Ordenar por fecha
datos = datos.sort_values(
    "fecha_entrega"
).reset_index(drop=True)

# Separar aproximadamente 80% desarrollo y 20% prueba
# Sin dividir un mismo día entre los conjuntos
fecha_corte = datos.loc[
    int(len(datos) * 0.80), "fecha_entrega"
].normalize()

desarrollo = datos[
    datos["fecha_entrega"] < fecha_corte
].copy()

prueba = datos[
    datos["fecha_entrega"] >= fecha_corte
].copy()

# Confirmar que se crearon ambos conjuntos
assert len(desarrollo) > 0
assert len(prueba) > 0

print("PREPARACIÓN DEL MODELO")
print("-" * 45)
print("Variables predictoras:", len(variables))
print("Registros de desarrollo:", len(desarrollo))
print("Registros de prueba:", len(prueba))
print("Fecha de corte:", fecha_corte)

print("\nRESULTADOS POR CONJUNTO")
for nombre, grupo in [
    ("Desarrollo", desarrollo),
    ("Prueba", prueba)
]:
    print(f"\n{nombre}:")
    print(grupo["cumple_plazo"].value_counts())
    print(
        "Tasa de cumplimiento:",
        round(grupo["cumple_plazo"].mean() * 100, 2),
        "%"
    )


In [0]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

# Reservar la parte más reciente del desarrollo
# para validar el primer modelo
corte_validacion = desarrollo.loc[
    int(len(desarrollo) * 0.75),
    "fecha_entrega"
].normalize()

entrenamiento = desarrollo[
    desarrollo["fecha_entrega"] < corte_validacion
]

validacion = desarrollo[
    desarrollo["fecha_entrega"] >= corte_validacion
]

X_train = entrenamiento[variables]
y_train = entrenamiento["cumple_plazo"]

X_val = validacion[variables]
y_val = validacion["cumple_plazo"]

# Preparación de variables numéricas
preparacion_numerica = Pipeline([
    ("faltantes", SimpleImputer(strategy="median")),
    ("escala", StandardScaler())
])

# Preparación de variables categóricas
preparacion_categorica = Pipeline([
    ("faltantes", SimpleImputer(
        strategy="constant",
        fill_value="Sin dato"
    )),
    ("codificacion", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

# Combinar ambas preparaciones
preprocesador = ColumnTransformer([
    ("numericas", preparacion_numerica, numericas),
    ("categoricas", preparacion_categorica, categoricas)
])

# Crear el primer modelo
modelo_logistico = Pipeline([
    ("preprocesador", preprocesador),
    ("clasificador", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

# Entrenar
modelo_logistico.fit(X_train, y_train)

# Calcular probabilidades de pagar a tiempo
probabilidades = modelo_logistico.predict_proba(X_val)[:, 1]

# Obtener predicciones
predicciones = modelo_logistico.predict(X_val)

# Evaluar
auc = roc_auc_score(y_val, probabilidades)
accuracy = accuracy_score(y_val, predicciones)

print("PRIMER MODELO: REGRESIÓN LOGÍSTICA")
print("-" * 45)
print("Entrenamiento:", len(X_train))
print("Validación:", len(X_val))
print(f"ROC-AUC: {auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")


In [0]:

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    average_precision_score
)
import pandas as pd

# Modelos que vamos a comparar
algoritmos = {
    "Regresion Logistica": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

resultados = []
modelos_entrenados = {}

# Entrenar cada algoritmo
for nombre, algoritmo in algoritmos.items():

    print(f"Entrenando: {nombre}")

    modelo = Pipeline([
        ("preprocesador", clone(preprocesador)),
        ("clasificador", algoritmo)
    ])

    modelo.fit(X_train, y_train)

    # Probabilidades de cumplir el plazo
    probabilidades = modelo.predict_proba(X_val)[:, 1]

    # Predicciones de clase
    predicciones = modelo.predict(X_val)

    # Calcular metricas
    auc = roc_auc_score(y_val, probabilidades)
    accuracy = accuracy_score(y_val, predicciones)
    ap = average_precision_score(
        y_val, probabilidades
    )

    resultados.append({
        "Modelo": nombre,
        "ROC_AUC": round(auc, 4),
        "Accuracy": round(accuracy, 4),
        "Average_Precision": round(ap, 4)
    })

    modelos_entrenados[nombre] = modelo

# Crear tabla comparativa
tabla_resultados = pd.DataFrame(resultados)

tabla_resultados = tabla_resultados.sort_values(
    "ROC_AUC", ascending=False
).reset_index(drop=True)

print("\nCOMPARACION DE MODELOS")
print("-" * 50)

display(tabla_resultados)

# Identificar el mejor en validacion
mejor_nombre = tabla_resultados.iloc[0]["Modelo"]
mejor_modelo = modelos_entrenados[mejor_nombre]

print("\nMejor modelo en validacion:", mejor_nombre)
print(
    "ROC-AUC:",
    tabla_resultados.iloc[0]["ROC_AUC"]
)


In [0]:

import matplotlib.pyplot as plt

# Ordenar resultados para visualizarlos
grafica = tabla_resultados.sort_values("ROC_AUC")

plt.figure(figsize=(10, 5))

barras = plt.barh(
    grafica["Modelo"],
    grafica["ROC_AUC"]
)

# Marcar el objetivo
plt.axvline(
    x=0.70,
    color="red",
    linestyle="--",
    label="Meta AUC = 0.70"
)

# Mostrar valor de cada modelo
for barra in barras:
    valor = barra.get_width()

    plt.text(
        valor + 0.01,
        barra.get_y() + barra.get_height()/2,
        f"{valor:.4f}",
        va="center"
    )

plt.xlim(0, 1)
plt.xlabel("ROC-AUC en validacion")
plt.title("Comparacion de modelos - ROGA Insight AI")
plt.legend()
plt.grid(axis="x", alpha=0.2)

plt.show()


In [0]:

import numpy as np

# Crear copias independientes
X_train_mejorado = X_train.copy()
X_val_mejorado = X_val.copy()

def crear_indicadores(datos):
    datos = datos.copy()

    # Porcentaje del pedido financiado
    datos["proporcion_financiada"] = (
        datos["monto_financiado"] /
        datos["Importe_Pedido"].replace(0, np.nan)
    )

    # Pago quincenal respecto al monto financiado
    datos["pago_sobre_financiamiento"] = (
        datos["pago_quincenal"] /
        datos["monto_financiado"].replace(0, np.nan)
    )

    # Identificar capacidades negativas
    datos["capacidad_negativa"] = (
        datos["Capacidad_Quincenal"] < 0
    ).astype(int)

    # Capacidad respecto al pago solicitado
    datos["capacidad_sobre_pago"] = (
        datos["Capacidad_Quincenal"] /
        datos["pago_quincenal"].replace(0, np.nan)
    )

    return datos.replace(
        [np.inf, -np.inf], np.nan
    )

X_train_mejorado = crear_indicadores(X_train_mejorado)
X_val_mejorado = crear_indicadores(X_val_mejorado)

print("Variables originales:", X_train.shape[1])
print("Variables mejoradas:", X_train_mejorado.shape[1])
print("Entrenamiento:", len(X_train_mejorado))
print("Validacion:", len(X_val_mejorado))


In [0]:

%pip install catboost

from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

# Preparar copias para CatBoost
X_train_cat = X_train_mejorado.copy()
X_val_cat = X_val_mejorado.copy()

# Identificar las columnas categoricas
columnas_cat = [
    c for c in categoricas
    if c in X_train_cat.columns
]

# CatBoost necesita categorias sin valores nulos
for columna in columnas_cat:
    X_train_cat[columna] = (
        X_train_cat[columna]
        .fillna("Sin dato")
        .astype(str)
    )
    X_val_cat[columna] = (
        X_val_cat[columna]
        .fillna("Sin dato")
        .astype(str)
    )

# Los valores numericos faltantes se conservan como NaN

modelo_catboost = CatBoostClassifier(
    iterations=800,
    depth=4,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False,
    thread_count=2
)

# Entrenar sin usar la validacion como datos de ajuste
modelo_catboost.fit(
    X_train_cat,
    y_train,
    cat_features=columnas_cat
)

# Predecir probabilidades
prob_catboost = modelo_catboost.predict_proba(
    X_val_cat
)[:, 1]

auc_catboost = roc_auc_score(
    y_val,
    prob_catboost
)

print("RESULTADO CATBOOST")
print("-" * 40)
print(f"ROC-AUC: {auc_catboost:.4f}")
print("Gradient Boosting anterior: 0.6255")
print(f"Diferencia: {auc_catboost - 0.6255:+.4f}")



In [0]:

import matplotlib.pyplot as plt
import pandas as pd

importancias = pd.DataFrame({
    "Variable": X_train_cat.columns,
    "Importancia": modelo_catboost.feature_importances_
})

importancias = importancias.sort_values(
    "Importancia",
    ascending=False
)

display(importancias)

top10 = importancias.head(10).sort_values("Importancia")

plt.figure(figsize=(10, 6))
plt.barh(
    top10["Variable"],
    top10["Importancia"]
)
plt.title("Variables más importantes - CatBoost")
plt.xlabel("Importancia relativa del modelo")
plt.tight_layout()
plt.show()


In [0]:

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# Trabajamos SOLO con desarrollo
datos_cv = desarrollo.sort_values(
    "fecha_entrega"
).copy()

# Preparar variables originales y mejoradas
X_original = datos_cv[variables].copy()
X_mejorado = crear_indicadores(X_original)

# Fechas únicas para no dividir un mismo día
dias = np.array(sorted(
    datos_cv["fecha_entrega"].dt.normalize().unique()
))

# Tres ventanas temporales
ventanas = [
    (0.50, 0.65),
    (0.65, 0.80),
    (0.80, 1.00)
]

resultados_cv = []

for nombre, X, nums in [
    ("Original", X_original, numericas),
    ("Mejorado", X_mejorado,
     numericas + [
         "proporcion_financiada",
         "pago_sobre_financiamiento",
         "capacidad_negativa",
         "capacidad_sobre_pago"
     ])
]:

    # Crear el preprocesamiento para cada variante
    prep = clone(preprocesador)

    # Actualizar columnas numericas
    prep.set_params(
        transformers=[
            ("numericas",
             clone(preparacion_numerica), nums),
            ("categoricas",
             clone(preparacion_categorica), categoricas)
        ]
    )

    for i, (inicio, fin) in enumerate(ventanas):

        fecha_inicio = dias[int(len(dias) * inicio)]
        fecha_fin = (
            dias[int(len(dias) * fin)]
            if fin < 1 else None
        )

        fechas = datos_cv["fecha_entrega"].dt.normalize()

        mask_train = fechas < fecha_inicio
        mask_val = fechas >= fecha_inicio

        if fecha_fin is not None:
            mask_val = mask_val & (fechas < fecha_fin)

        y_ent = datos_cv.loc[mask_train, "cumple_plazo"]
        y_valid = datos_cv.loc[mask_val, "cumple_plazo"]

        modelo = Pipeline([
            ("prep", clone(prep)),
            ("modelo", GradientBoostingClassifier(
                n_estimators=100,
                learning_rate=0.05,
                max_depth=3,
                random_state=42
            ))
        ])

        modelo.fit(X.loc[mask_train], y_ent)

        prob = modelo.predict_proba(
            X.loc[mask_val]
        )[:, 1]

        auc = roc_auc_score(y_valid, prob)

        resultados_cv.append({
            "Variables": nombre,
            "Ventana": i + 1,
            "Entrenamiento": mask_train.sum(),
            "Validacion": mask_val.sum(),
            "AUC": round(auc, 4)
        })

tabla_cv = pd.DataFrame(resultados_cv)

print("VALIDACION TEMPORAL")
display(tabla_cv)

print("\nPROMEDIOS POR VERSION")
display(
    tabla_cv.groupby("Variables")["AUC"]
    .agg(["mean", "std"])
    .round(4)
)


In [0]:

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Recuperar variables originales
X_desarrollo = desarrollo[variables]
y_desarrollo = desarrollo["cumple_plazo"]

X_prueba = prueba[variables]
y_prueba = prueba["cumple_plazo"]

# Modelo definitivo
modelo_final = Pipeline([
    ("preprocesador", clone(preprocesador)),
    ("clasificador", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

# Entrenar con todo el conjunto de desarrollo
modelo_final.fit(X_desarrollo, y_desarrollo)

# Evaluación final
probabilidades_finales = modelo_final.predict_proba(
    X_prueba
)[:, 1]

predicciones_finales = modelo_final.predict(X_prueba)

auc_final = roc_auc_score(
    y_prueba,
    probabilidades_finales
)

accuracy_final = accuracy_score(
    y_prueba,
    predicciones_finales
)

print("EVALUACIÓN FINAL - ROGA INSIGHT AI")
print("-" * 45)

print("Registros de desarrollo:", len(X_desarrollo))
print("Registros de prueba:", len(X_prueba))

print(f"\nROC-AUC final: {auc_final:.4f}")
print(f"Accuracy final: {accuracy_final:.4f}")

print("\nREPORTE DE CLASIFICACIÓN")
print(classification_report(
    y_prueba,
    predicciones_finales,
    zero_division=0
))

print("\nMATRIZ DE CONFUSIÓN")
print(confusion_matrix(
    y_prueba,
    predicciones_finales
))


In [0]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score
)

# Utilizamos el modelo de Gradient Boosting
# entrenado antes de la prueba final
modelo_validacion = modelos_entrenados["Gradient Boosting"]

# Probabilidad de pagar a tiempo
prob_cumplimiento = modelo_validacion.predict_proba(
    X_val
)[:, 1]

# Convertimos a probabilidad de NO pagar a tiempo
prob_riesgo = 1 - prob_cumplimiento

# Resultado real: 1 significa fuera de plazo
y_riesgo = (y_val == 0).astype(int)

resultados_umbrales = []

for umbral in [0.30, 0.40, 0.50, 0.60, 0.70]:

    pred_riesgo = (
        prob_riesgo >= umbral
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_riesgo,
        pred_riesgo,
        labels=[0, 1]
    ).ravel()

    resultados_umbrales.append({
        "Umbral": umbral,
        "Riesgos_detectados": tp,
        "Riesgos_no_detectados": fn,
        "Clientes_cumplidos_alertados": fp,
        "Recall_riesgo": round(
            recall_score(y_riesgo, pred_riesgo), 3
        ),
        "Precision_riesgo": round(
            precision_score(
                y_riesgo,
                pred_riesgo,
                zero_division=0
            ), 3
        )
    })

tabla_umbrales = pd.DataFrame(resultados_umbrales)

display(tabla_umbrales)


In [0]:

import os
import json
import joblib

# Crear carpeta para los archivos del modelo
carpeta_modelos = "modelos"
os.makedirs(carpeta_modelos, exist_ok=True)

# Guardar el modelo final
ruta_modelo = os.path.join(
    carpeta_modelos,
    "roga_gradient_boosting.joblib"
)

joblib.dump(modelo_final, ruta_modelo)

# Guardar configuracion y resultados
configuracion = {
    "nombre": "ROGA Insight AI",
    "version": "1.0-prototipo",
    "poblacion": "Cliente Nuevo",
    "variables": variables,
    "umbral_alerta": 0.40,
    "roc_auc_prueba": float(auc_final),
    "accuracy_prueba": float(accuracy_final),
    "registros_desarrollo": len(X_desarrollo),
    "registros_prueba": len(X_prueba),
    "advertencia": (
        "Prototipo de apoyo al analista. "
        "No utilizar para decisiones automaticas."
    )
}

ruta_config = os.path.join(
    carpeta_modelos,
    "configuracion.json"
)

with open(ruta_config, "w", encoding="utf-8") as archivo:
    json.dump(
        configuracion,
        archivo,
        indent=4,
        ensure_ascii=False
    )

print("MODELO GUARDADO")
print("Archivo:", ruta_modelo)
print("Configuracion:", ruta_config)

# Comprobar que el modelo puede volver a cargarse
modelo_cargado = joblib.load(ruta_modelo)

print("\nVerificacion de carga: correcta")
